In [1]:
!pip install polars emoji nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.7/805.7 kB 35.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 MB 214.2 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [polars]2m1/2 [polars]


In [ ]:
import polars as pl
import emoji
import re
import nltk
from nltk.corpus import stopwords
import sys

# 1. Setup Stopwords
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english')) - {
    "not", "no", "never", "n't", "none", "neither", "nowhere", "nothing", 
    "broken", "damage", "defective", "return", "refund", "waste", "worst"
}

# --- PROGRESS TRACKING SETUP ---
# We use a global counter to track how many rows the Python function has processed
processed_count = 0
total_rows = 0 

# 2. Define Python Cleaning Function (Modified for Progress Updates)
def demojize_text(text):
    global processed_count, total_rows
    
    # Update progress every 50,000 rows to avoid flooding the output
    processed_count += 1
    if processed_count % 50000 == 0:
        # \r allows us to overwrite the same line (like a progress bar)
        sys.stdout.write(f"\rProcessed: {processed_count} / {total_rows} rows...")
        sys.stdout.flush()

    if text is None:
        return ""
    return emoji.demojize(text, delimiters=(" ", " ")).replace("_", " ")

# 3. Load Data
try:
    s3_path = "s3://amazon-project-dataset-meta-review/raw/reviews/health_household_reviews_filtered.jsonl"
    lf = pl.scan_ndjson(s3_path)
    
    # --- GET TOTAL ROWS FIRST ---
    # We run a quick count so we know the 100% mark for our progress bar
    print("Counting total rows...")
    total_rows = lf.select(pl.len()).collect().item()
    print(f"Total rows to process: {total_rows}")
    
except Exception as e:
    print(f"Could not load S3 data: {e}")
    print("Loading sample data for demonstration...")
    data = {
        "asin": ["A123", "B456"],
        "parent_asin": ["P1", "P2"],
        "rating": [5.0, 1.0],
        "title": ["Great 🎸", "Bad!"],
        "text": ["Loved it! 😊", "Do not buy. http://spam.com"],
        "helpful_vote": [0, 1],
        "timestamp": [123, 124],
        "verified_purchase": [True, False]
    }
    lf = pl.LazyFrame(data)
    total_rows = 2

# 4. The Cleaning Pipeline (Logic Unchanged)
print("Starting processing pipeline...")
final_df = (
    lf
    .with_columns([
        pl.col("title").fill_null(""),
        pl.col("text").fill_null("")
    ])
    .with_columns(
        raw_combined = pl.concat_str([pl.col("title"), pl.col("text")], separator=" ")
    )
    # This is where the progress bar will update
    .with_columns(
        text_emojized = pl.col("raw_combined").map_elements(demojize_text, return_dtype=pl.Utf8)
    )
    .with_columns(
        text_clean = pl.col("text_emojized")
        .str.to_lowercase()
        .str.replace_all(r"http\S+", "")
        .str.replace_all(r"[^a-z\s]", "")
        .str.strip_chars()
    )
    .with_columns(
        final_text = pl.col("text_clean")
        .str.split(" ")
        .list.eval(
            pl.element().filter(~pl.element().is_in(stop_words))
        )
        .list.join(" ")
    )
    .select(["parent_asin","asin", "rating", "final_text","helpful_vote","timestamp","verified_purchase"])
    .collect()
)

print(f"\nProcessing Complete! Final shape: {final_df.shape}")

# 5. Show Output
print(final_df.head())

# --- 6. SAVE TO S3 (New Addition) ---
# We define a new path for the cleaned parquet file


Counting total rows...
Total rows to process: 14245530
Starting processing pipeline...


In [41]:
o = "s3://curated-review-data/meta/*.parquet"
lfo = pl.scan_parquet(o)

# print("--- Schema ---")
# print(lfo.collect_schema())

# total_rows = lfo.select(pl.len()).collect().item()
# print(f"\n--- Total Rows: {total_rows} ---")

# print("\n--- Null Counts ---")
# print(lfo.null_count().collect())
stats = lfo.select([
    pl.len().alias("total_rows"),
    pl.col("parent_asin").n_unique().alias("unique_parent_asins")
]).collect()
print(stats)

shape: (1, 2)
┌────────────┬─────────────────────┐
│ total_rows ┆ unique_parent_asins │
│ ---        ┆ ---                 │
│ u32        ┆ u32                 │
╞════════════╪═════════════════════╡
│ 1382399    ┆ 1382399             │
└────────────┴─────────────────────┘


In [36]:
titles = final_df.select("final_text").sample(20).to_series().to_list()

for i, title in enumerate(titles, 1):
    print(f"{i}. {title}\n" + "-"*30)

1. seems well made skeptical spending  seems well made time tell good buy guessbr made china wondering
------------------------------
2. soft need row bamboo cleaning toothbrushes soft bought bamboo brushes row charcoal cleaned teeth far better
------------------------------
3. mehhhh wrapping paper looked like embossed photos definitely looks dramatically different person wouldnt paid  ok wrapped wedding present  multiple wrapping papers got target much cuter way nicer quality
------------------------------
4. not work not work thought would chewable wouldnt waste money leaves taste also taste mint
------------------------------
5. comfortable comfortable block pretty much sound hear cars driving night loud people upstairs dogs barking name
------------------------------
6. get paid quality usually not good cheap
------------------------------
7. calmed ibs saw collagen promoted health infomercial  collagen expensive  looked amazon found collagen much better pricebr  days use calmed i

In [25]:
with pl.Config(fmt_str_lengths=1000, tbl_width_chars=1000):
    # Sample 20 random rows and display just the title
    # (Using your existing DataFrame 'final_df')
    print(final_df.select("final_text").sample(20))

shape: (20, 1)
┌───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ final_text                                                                                                                                                                                                                                                                                                                                        │
│ ---                                                                                                                                                                                                                                                                                                        

In [26]:
output_path = "s3://curated-review-data/reviews/health_household_reviews.parquet"

print(f"Saving data to {output_path}...")
try:
    # write_parquet automatically handles S3 if s3fs is installed
    final_df.write_parquet(output_path)
    print("Save Successful!")
except Exception as e:
    print(f"Error saving to S3: {e}")

Saving data to s3://curated-review-data/reviews/fashion_reviews.parquet...
Save Successful!


In [19]:

lf1 = pl.scan_parquet(output_path)

print("--- Schema ---")
print(lf1.collect_schema())

total_rows = lf1.select(pl.len()).collect().item()
print(f"\n--- Total Rows: {total_rows} ---")

print("\n--- Null Counts ---")
print(lf1.null_count().collect())

--- Schema ---
Schema({'parent_asin': String, 'asin': String, 'rating': Float64, 'final_text': String, 'helpful_vote': Int64, 'timestamp': Int64, 'verified_purchase': Boolean})

--- Total Rows: 1978085 ---

--- Null Counts ---
shape: (1, 7)
┌─────────────┬──────┬────────┬────────────┬──────────────┬───────────┬───────────────────┐
│ parent_asin ┆ asin ┆ rating ┆ final_text ┆ helpful_vote ┆ timestamp ┆ verified_purchase │
│ ---         ┆ ---  ┆ ---    ┆ ---        ┆ ---          ┆ ---       ┆ ---               │
│ u32         ┆ u32  ┆ u32    ┆ u32        ┆ u32          ┆ u32       ┆ u32               │
╞═════════════╪══════╪════════╪════════════╪══════════════╪═══════════╪═══════════════════╡
│ 0           ┆ 0    ┆ 0      ┆ 0          ┆ 0            ┆ 0         ┆ 0                 │
└─────────────┴──────┴────────┴────────────┴──────────────┴───────────┴───────────────────┘


In [20]:
print(lf1.head(10).collect())

shape: (10, 7)
┌─────────────┬────────────┬────────┬───────────────┬──────────────┬───────────────┬───────────────┐
│ parent_asin ┆ asin       ┆ rating ┆ final_text    ┆ helpful_vote ┆ timestamp     ┆ verified_purc │
│ ---         ┆ ---        ┆ ---    ┆ ---           ┆ ---          ┆ ---           ┆ hase          │
│ str         ┆ str        ┆ f64    ┆ str           ┆ i64          ┆ i64           ┆ ---           │
│             ┆            ┆        ┆               ┆              ┆               ┆ bool          │
╞═════════════╪════════════╪════════╪═══════════════╪══════════════╪═══════════════╪═══════════════╡
│ B01N0TQ0OH  ┆ B01N0TQ0OH ┆ 5.0    ┆ work great    ┆ 0            ┆ 1519317108692 ┆ true          │
│             ┆            ┆        ┆ work great    ┆              ┆               ┆               │
│             ┆            ┆        ┆ use new …     ┆              ┆               ┆               │
│ B07DD37QPZ  ┆ B07DD2DMXB ┆ 5.0    ┆ excellent     ┆ 0            ┆ 1664746